### Соревнование по Multi‑Armed Bandit (MAB)!

Вы будете разрабатывать агента, который в каждом раунде выбирает один из 100 игровых автоматов (бандитов) и получает награду (0 или 1). Ваша цель – максимизировать суммарную награду за всю игру.

Как устроена среда

- Количество бандитов: 100 (пронумерованы от 0 до 99).

- Длительность одного эпизода: 2000 раундов (в каждом раунде оба агента последовательно делают выбор). Для итогового результата будет сыграно множество эпизодов.

- Вероятности: каждый бандит имеет свою начальную вероятность выигрыша (генерируется случайно из бета‑распределения).
    Важно: при каждом выборе бандита его вероятность уменьшается на 3% (умножается на 0.97). Это означает, что чем чаще вы дёргаете один и тот же рычаг, тем менее выгодным он становится.

- Буст: после каждого раунда с некоторой вероятностью (случайно) один из бандитов получает небольшое увеличение вероятности (имитация изменения внешних условий).

- Два игрока: в каждом эпизоде соревнуются два агента. Они видят все действия друг друга, но не видят наград противника – только свои собственные.

- Порядок хода: в каждом раунде агенты ходят последовательно (первый игрок дёргает рычаг, затем второй). Чтобы исключить преимущество первого хода, в разных эпизодах порядок меняется (чётный эпизод – агент0 первый, нечётный – агент1 первый). Ваш агент получает информацию о своей позиции (position в configuration).

### Что нужно:

Вам нужно написать функцию на Python с именем agent, которая принимает два аргумента:

In [1]:
def agent(observation, configuration):
    # ... ваш код ...
    return action  # целое число от 0 до num_bandits-1

**Входные данные**

`observation` – словарь, содержащий:

 - `n_pulls` – массив (список) длины num_bandits, где n_pulls[i] – сколько раз вы дёргали бандит i за все прошлые раунды.

 - `sum_rewards` – массив (список) длины num_bandits, где sum_rewards[i] – сумма наград, полученных вами от бандита i за все прошлые раунды.

 - `step` – текущий номер раунда (начинается с 0 и до num_rounds-1).

 - `my_actions` – список ваших действий (номеров бандитов) за все прошлые раунды (в хронологическом порядке).

 - `other_actions` – список действий другого агента за все прошлые раунды.

`configuration` – словарь, содержащий:

 - `num_bandits` – общее число бандитов.

 - `num_rounds` – общее число раундов.

 - `position` – ваша позиция в текущем эпизоде: 0 – вы ходите первым, 1 – вторым.

**Выходные данные**

Функция должна вернуть целое число от 0 до `num_bandits` - 1 – номер бандита, которого вы выбираете в текущем раунде.

### Как предоставить решение

    Создайте файл с названием типа (например, ФамилияИО_группа.py) и поместите в него функцию agent.

    Файл должен быть самодостаточным – все импорты (numpy, random, math и т.д.) делайте внутри файла.

Файл присылайте на почту: a.krivoshein@spbu.ru

# Пример с распечаткой

In [2]:
# main.py
import os
import importlib
import numpy as np
from mab_env import DynamicMABEnv, evaluate_agents, run_episode  # предполагаем, что класс среды вынесен в отдельный файл
import random

def random_agent(observation, configuration):
    """Случайный выбор бандита."""
    return random.randrange(configuration['num_bandits'])
    
# Агент-обёртка для вывода всех данных
def print_agent(observation, configuration):
    print("\n" + "="*50)
    print(f"Агент (position={configuration['position']}) получил:")
    print("  observation:")
    for key, value in observation.items():
        if isinstance(value, list):
            print(f"    {key}: {value}")
        elif isinstance(value, np.ndarray):
            print(f"    {key}: {value.tolist()}")
        else:
            print(f"    {key}: {value}")
    print("  configuration:")
    for key, value in configuration.items():
        print(f"    {key}: {value}")
    print("="*50)
    
    # Вызываем базовую стратегию (например, случайную или жадную)
    # Здесь можно заменить на любого другого агента
    return random_agent(observation, configuration)

# Создаём среду с 5 бандитами на 5 раундов (чтобы было немного данных)
env = DynamicMABEnv(num_bandits=5, num_rounds=5, seed=42, boost_prob=0.0)

print("=== Эпизод с episode_idx=0 (стандартный порядок: агент0 -> агент1) ===")
total0 = run_episode([print_agent, print_agent], env, episode_idx=0, verbose=False)
print(f"\nИтоговые награды: агент0 = {total0[0]}, агент1 = {total0[1]}\n")

# Создаём новую среду с тем же seed для второго эпизода
env2 = DynamicMABEnv(num_bandits=5, num_rounds=5, seed=42, boost_prob=0.0)

print("=== Эпизод с episode_idx=1 (переставленный порядок: агент1 -> агент0) ===")
total1 = run_episode([print_agent, print_agent], env2, episode_idx=1, verbose=False)
print(f"\nИтоговые награды: агент0 = {total1[0]}, агент1 = {total1[1]}")

=== Эпизод с episode_idx=0 (стандартный порядок: агент0 -> агент1) ===

Агент (position=0) получил:
  observation:
    n_pulls: [0, 0, 0, 0, 0]
    sum_rewards: [0, 0, 0, 0, 0]
    step: 0
    my_actions: []
    other_actions: []
  configuration:
    num_bandits: 5
    num_rounds: 5
    position: 0

Агент (position=1) получил:
  observation:
    n_pulls: [0, 0, 0, 0, 0]
    sum_rewards: [0, 0, 0, 0, 0]
    step: 0
    my_actions: []
    other_actions: []
  configuration:
    num_bandits: 5
    num_rounds: 5
    position: 1

Агент (position=0) получил:
  observation:
    n_pulls: [1, 0, 0, 0, 0]
    sum_rewards: [1, 0, 0, 0, 0]
    step: 1
    my_actions: [0]
    other_actions: [0]
  configuration:
    num_bandits: 5
    num_rounds: 5
    position: 0

Агент (position=1) получил:
  observation:
    n_pulls: [1, 0, 0, 0, 0]
    sum_rewards: [1, 0, 0, 0, 0]
    step: 1
    my_actions: [0]
    other_actions: [0]
  configuration:
    num_bandits: 5
    num_rounds: 5
    position: 1

Агент (p

In [3]:
# Путь к папке с агентами
AGENTS_DIR = "agents"

def load_agents_from_folder(folder_path):
    """
    Загружает все .py файлы из указанной папки и возвращает словарь:
    {имя_модуля: функция_агента}
    """
    agents = {}
    # Проходим по всем файлам в папке
    for filename in os.listdir(folder_path):
        if filename.endswith(".py") and not filename.startswith("__"):
            module_name = filename[:-3]  # убираем .py
            # Динамический импорт
            spec = importlib.util.spec_from_file_location(module_name, os.path.join(folder_path, filename))
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            # Проверяем, есть ли функция 'agent'
            if hasattr(module, 'agent'):
                agents[module_name] = module.agent
            else:
                print(f"Предупреждение: в файле {filename} не найдена функция 'agent'")
    return agents


all_agents = load_agents_from_folder(AGENTS_DIR)
agent_names = list(all_agents.keys())
print(f"Загружено агентов: {len(all_agents)}: {agent_names}")

    # Генерируем все возможные пары (каждый против каждого)
agent_pairs = []
for i, name1 in enumerate(agent_names):
    for j, name2 in enumerate(agent_names):
        if i < j:  # каждая пара только один раз
            agent_pairs.append((name1, all_agents[name1], name2, all_agents[name2]))

Загружено агентов: 2: ['ЖадныйИО_00', 'СлучайныйИО_00']


In [6]:
# Параметры 
NUM_BANDITS = 100
NUM_ROUNDS = 2000
NUM_EPISODES = 30   

print("Запуск турнира (динамические вероятности)...")
results = evaluate_agents(agent_pairs,
                              num_bandits=NUM_BANDITS,
                              num_rounds=NUM_ROUNDS,
                              num_episodes=NUM_EPISODES,
                              seed=42)

print("\n=== РЕЗУЛЬТАТЫ ТУРНИРА ===")
for pair, stats in results.items():
    names = list(stats.keys())
    print(f"\n{pair}:")
    print(f"  {names[0]}: {stats[names[0]]:.2f}")
    print(f"  {names[1]}: {stats[names[1]]:.2f}")
    print(f"  Разница: {stats['diff']:+.2f}")

Запуск турнира (динамические вероятности)...


100%|██████████| 30/30 [00:01<00:00, 17.13it/s]


=== РЕЗУЛЬТАТЫ ТУРНИРА ===

ЖадныйИО_00 vs СлучайныйИО_00:
  ЖадныйИО_00: 593.73
  СлучайныйИО_00: 555.70
  Разница: +38.03
